In [17]:
!pip install torchinfo

In [1]:
import torch
from torch import nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import torch.optim as optim
from torchinfo import summary

In [9]:
# 공개 데이터셋에서 학습 데이터를 다운
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# 공개 데이터셋에서 테스트 데이터를 다운
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

batch_size = 32
# 데이터로더 생성 
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)


In [2]:
device = ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using {device} device")

class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, padding='same'), #(b,1,28,28)->(b,6,28,28)
            nn.Sigmoid(),
            nn.AvgPool2d(kernel_size=2, stride=2), #(b,6,14,14)
            nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5), #(b,16,10,10)
            nn.Sigmoid(),
            nn.AvgPool2d(kernel_size=2, stride=2) #(b,16,5,5)
        )
                
        self.classifier = nn.Sequential(
            nn.Linear(in_features=16 * 5 * 5, out_features=120), #(400,120)
            nn.Sigmoid(),
            nn.Linear(in_features=120, out_features=84),
            nn.Sigmoid(),
            nn.Linear(in_features=84, out_features=10)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x,1)
        x = self.classifier(x)
        return x

lenet5 = LeNet5().to(device)
print(lenet5)

Using cuda device
LeNet5(
  (features): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=same)
    (1): Sigmoid()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (4): Sigmoid()
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
  )
  (classifier): Sequential(
    (0): Linear(in_features=400, out_features=120, bias=True)
    (1): Sigmoid()
    (2): Linear(in_features=120, out_features=84, bias=True)
    (3): Sigmoid()
    (4): Linear(in_features=84, out_features=10, bias=True)
  )
)


In [3]:
summary(lenet5, input_size=(32, 1, 28, 28))

Layer (type:depth-idx)                   Output Shape              Param #
LeNet5                                   [32, 10]                  --
├─Sequential: 1-1                        [32, 16, 5, 5]            --
│    └─Conv2d: 2-1                       [32, 6, 28, 28]           156
│    └─Sigmoid: 2-2                      [32, 6, 28, 28]           --
│    └─AvgPool2d: 2-3                    [32, 6, 14, 14]           --
│    └─Conv2d: 2-4                       [32, 16, 10, 10]          2,416
│    └─Sigmoid: 2-5                      [32, 16, 10, 10]          --
│    └─AvgPool2d: 2-6                    [32, 16, 5, 5]            --
├─Sequential: 1-2                        [32, 10]                  --
│    └─Linear: 2-7                       [32, 120]                 48,120
│    └─Sigmoid: 2-8                      [32, 120]                 --
│    └─Linear: 2-9                       [32, 84]                  10,164
│    └─Sigmoid: 2-10                     [32, 84]                  --
│  

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lenet5.parameters(), lr=0.001)

In [13]:
def train(dataloader, lenet5, criterion, optimizer):
    size = len(dataloader.dataset)
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # 예측 오류 계산
        pred = lenet5(X)
        loss = criterion(pred, y)

        # 역전파
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test(dataloader, lenet5, criterion):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    lenet5.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = lenet5(X)
            test_loss += criterion(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [30]:
epochs = 13
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, lenet5, criterion, optimizer)
    test(test_dataloader, lenet5, criterion)
print("끝나브러써!")

Epoch 1
-------------------------------
loss: 2.265234  [   32/60000]
loss: 2.262707  [ 3232/60000]
loss: 2.380144  [ 6432/60000]
loss: 2.424652  [ 9632/60000]
loss: 2.406330  [12832/60000]
loss: 2.354134  [16032/60000]
loss: 2.297873  [19232/60000]
loss: 2.329626  [22432/60000]
loss: 2.324889  [25632/60000]
loss: 2.402275  [28832/60000]
loss: 2.363238  [32032/60000]
loss: 2.339233  [35232/60000]
loss: 2.246435  [38432/60000]
loss: 2.351878  [41632/60000]
loss: 2.289105  [44832/60000]
loss: 2.319295  [48032/60000]
loss: 2.270716  [51232/60000]
loss: 2.335484  [54432/60000]
loss: 2.381771  [57632/60000]
Test Error: 
 Accuracy: 10.0%, Avg loss: 2.345529 

Epoch 2
-------------------------------
loss: 2.265234  [   32/60000]
loss: 2.262707  [ 3232/60000]
loss: 2.380144  [ 6432/60000]
loss: 2.424652  [ 9632/60000]
loss: 2.406330  [12832/60000]
loss: 2.354134  [16032/60000]
loss: 2.297873  [19232/60000]
loss: 2.329626  [22432/60000]
loss: 2.324889  [25632/60000]
loss: 2.402275  [28832/60000